# Ponder Lab Workflow Debug Notebook

Run each stage **in order** to inspect:
- Google Sheet history loaded into context
- **System instruction** (YAML prompts via `load_stage_prompt`)
- **User prompt** (dynamic vibe / selected concept / history JSON)
- **Response schema** sent to Gemini structured output
- **Raw LLM JSON** and **parsed Pydantic** result

> Uses only `app.*` modules — same code paths as the FastAPI routes.

**Setup (once):**
```bash
cd backend
pip install -r requirements.txt -r requirements-notebooks.txt
jupyter notebook notebooks/workflow_debug.ipynb
```

In [4]:
# Cell 1 — Environment & imports
import json
import sys
from pathlib import Path

# Toggle deep prompt/raw dumps. Keep False for concise stage-by-stage decisions.
VERBOSE = True

# Resolve backend root whether notebook is opened from backend/ or backend/notebooks/
NOTEBOOK_DIR = Path.cwd()
if (NOTEBOOK_DIR / "app").exists():
    BACKEND_ROOT = NOTEBOOK_DIR
elif (NOTEBOOK_DIR.parent / "app").exists():
    BACKEND_ROOT = NOTEBOOK_DIR.parent
else:
    raise RuntimeError("Run this notebook from backend/ or backend/notebooks/")

if str(BACKEND_ROOT) not in sys.path:
    sys.path.insert(0, str(BACKEND_ROOT))

print("BACKEND_ROOT:", BACKEND_ROOT)
print("sys.path[0]:", sys.path[0])
print("VERBOSE:", VERBOSE)

from app.core.config import get_settings
from app.schemas.llm_output import (
    IdeasLLMOutput,
    MetadataLLMOutput,
    ThumbnailLLMOutput,
    TitlesLLMOutput,
)
from app.services.gemini import get_gemini_service
from app.services.google_sheets import get_sheets_service
from app.utils.notebook_debug import print_banner, print_block, print_llm_exchange
from app.utils.prompt_loader import load_shared_system_prompt, load_stage_prompt

settings = get_settings()
print_banner("Runtime configuration")
print_block("Gemini model", settings.gemini_model)
print_block("Spreadsheet ID", settings.spreadsheet_id)
print_block("Worksheet name", settings.worksheet_name)
print_block("Service account path", str(settings.service_account_path))
print_block("Service account exists", settings.service_account_path.exists())

BACKEND_ROOT: c:\Users\Harivarshan.Rajesh\Documents\01_work_files\02_RnD_Trying\33_ponder_lab_architect\backend
sys.path[0]: c:\Users\Harivarshan.Rajesh\Documents\01_work_files\02_RnD_Trying\33_ponder_lab_architect\backend
VERBOSE: True

  Runtime configuration

--- Gemini model ---
gemini-2.5-flash


--- Spreadsheet ID ---
1j9I7hpIaTxS3RFMlfL4gmr-6WGmgHLb9pV2gcTJpSks


--- Worksheet name ---
Ponder Lab Video Memory Repository


--- Service account path ---
C:\Users\Harivarshan.Rajesh\Documents\01_work_files\02_RnD_Trying\33_ponder_lab_architect\backend\service_account.json


--- Service account exists ---
True



## Stage 0 — Google Sheets history

Mirrors what `ideas.py` and `titles.py` read before calling Gemini.

In [5]:
# Cell 2 — Load sheet history (all rows for knowledgebase use)
HISTORY_LIMIT = None  # None = use full repository history

sheets = get_sheets_service()
history = sheets.get_recent_history(limit=HISTORY_LIMIT)
history_context = sheets.history_as_context(history)

print_banner("Stage 0 · Google Sheets history")
print_block("Rows fetched", len(history))
if VERBOSE:
    print_block("Normalized history objects", history)
    print_block("history_context string (injected into LLM user prompt)", history_context)
else:
    print_block("First history row preview", history[0] if history else {})
    print("Set VERBOSE = True in Cell 1 to print full history context.")


  Stage 0 · Google Sheets history

--- Rows fetched ---
7


--- Normalized history objects ---
[
  {
    "video_idea": "Pink or brownish noises in the base frequencies (bassy with a pinking feel) mixed with group chattering sounds in the higher frequencies. Goal is a silent crowd feel, giving the sensation of dissolving into the crowd.",
    "title": "MURMUR BROWN NOISE | Distant Chatter & Rumble | Cognitive Isolation | 10 Hrs | Deep Sleep",
    "description": "Isolate yourself from distractions with [10 Hours] of engineered Murmur Brown Noise. This track blends a deep, custom low-frequency rumble with the organic hum of distant human chatter, specifically balanced to provide \"social presence\" without semantic distraction—perfect for triggering the \"coffee shop effect\" to boost productivity or comfort a lonely mind for sleep.\n\nThe \"Ponder Lab\" Specification:\n\nAudio Type: Organic Brown Noise with Muffled Crowd Texture\nSound Design: Designed by Ponder Lab\nVisual: Pure Dark S

## Stage 1 — Ideas (`POST /api/v1/ideas/generate`)

Same prompt construction as `app/api/v1/ideas.py`.

In [6]:
# Cell 3 — Ideas generation
VIBE = "deep REM sleep, layered pink+brown, cave-like warmth, 8 hours"

ideas_system = load_stage_prompt("ideas")
ideas_user = (
    f"Vibe request: {VIBE}\n\n"
    "Repository Video Idea examples (match this prose style; do not repeat these mixes):\n"
    f"{sheets.video_idea_knowledgebase(history)}"
)

gemini = get_gemini_service()
ideas_parsed, ideas_raw = gemini.generate_structured_debug(
    ideas_system,
    ideas_user,
    IdeasLLMOutput,
)

print_banner("Stage 1 · Video idea choices (NOT titles)")
for i, concept in enumerate(ideas_parsed.concepts, start=1):
    print_block(
        f"Idea {i}: {concept.concept_name} ({concept.use_case}, {concept.duration_hours}h)",
        {
            "video_idea": concept.video_idea,
            "noise_stack": concept.noise_stack,
            "audience_benefit_reasoning": concept.audience_benefit_reasoning,
            "differentiator": concept.differentiator,
        },
    )

if VERBOSE:
    print_llm_exchange(
        stage="ideas",
        system_instruction=ideas_system,
        user_prompt=ideas_user,
        response_schema=IdeasLLMOutput,
        parsed=ideas_parsed,
        raw_text=ideas_raw,
        temperature=0.9,
        model_name=settings.gemini_model,
    )

selected_concept = ideas_parsed.concepts[0].model_dump()
print_banner("Stage 1 · Selection for next stage")
print_block("Auto-selected concept (index 0)", selected_concept)
print("Tip: Set selected_concept = ideas_parsed.concepts[n].model_dump() to choose manually.")

ServerError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}

## Stage 2 — Titles (`POST /api/v1/titles/generate`)

Same prompt construction as `app/api/v1/titles.py`.

In [ ]:
# Cell 4 — Titles generation
if "selected_concept" not in globals():
    raise RuntimeError("Run Stage 1 (ideas cell) first to define selected_concept")

titles_system = load_stage_prompt("titles")
titles_user = (
    f"Selected video idea (from concept stage):\n{json.dumps(selected_concept, indent=2)}\n\n"
    "Repository Title examples (match this bracketed pipe-separated style):\n"
    f"{sheets.title_knowledgebase(history)}"
)

titles_parsed, titles_raw = gemini.generate_structured_debug(
    titles_system,
    titles_user,
    TitlesLLMOutput,
    temperature=0.85,
)

print_banner("Stage 2 · YouTube title options")
for i, t in enumerate(titles_parsed.titles, start=1):
    print(f"{i}. {t.text}")

if VERBOSE:
    print_llm_exchange(
        stage="titles",
        system_instruction=titles_system,
        user_prompt=titles_user,
        response_schema=TitlesLLMOutput,
        parsed=titles_parsed,
        raw_text=titles_raw,
        temperature=0.85,
        model_name=settings.gemini_model,
    )

selected_title = titles_parsed.titles[0].text
print_banner("Stage 2 · Selection for next stage")
print_block("Auto-selected title (index 0)", selected_title)
print("Tip: Set selected_title = titles_parsed.titles[n].text to choose manually.")

## Stage 3 — Metadata (`POST /api/v1/metadata/generate`)

Uses full repository history as a style knowledgebase (including description formatting patterns).

In [ ]:
# Cell 5 — Metadata generation
from app.api.v1.metadata import _normalize_tags, _sanitize_description

if "selected_concept" not in globals() or "selected_title" not in globals():
    raise RuntimeError("Run Stage 1 and Stage 2 first")

metadata_system = load_stage_prompt("metadata")
metadata_user = (
    f"Selected concept:\n{json.dumps(selected_concept, indent=2)}\n\n"
    f"Approved title: {selected_title}\n\n"
    "Repository history descriptions for formatting style transfer:\n"
    f"{history_context}"
)

metadata_parsed, metadata_raw = gemini.generate_structured_debug(
    metadata_system,
    metadata_user,
    MetadataLLMOutput,
    temperature=0.7,
)

final_description = _sanitize_description(metadata_parsed.description)
final_tags = _normalize_tags(metadata_parsed.tags)

print_banner("Stage 3 · Final metadata (copy-paste ready)")
print_block("Description", final_description)
print_block("Tags", final_tags)
print_block("Tag count", len(final_tags))

if VERBOSE:
    print_llm_exchange(
        stage="metadata",
        system_instruction=metadata_system,
        user_prompt=metadata_user,
        response_schema=MetadataLLMOutput,
        parsed=metadata_parsed,
        raw_text=metadata_raw,
        temperature=0.7,
        model_name=settings.gemini_model,
    )

## Stage 4 — Thumbnail prompt (`POST /api/v1/metadata/thumbnail`)

In [ ]:
# Cell 6 — Thumbnail prompt generation
if "final_description" not in globals() or "final_tags" not in globals():
    raise RuntimeError("Run Stage 3 (metadata cell) first")

thumbnail_system = load_stage_prompt("thumbnail")
thumbnail_user = json.dumps(
    {
        "idea": selected_concept,
        "title": selected_title,
        "description": final_description,
        "tags": final_tags,
    },
    indent=2,
)

print_banner("Stage 4 · Prompt assembly (before LLM call)")
print_block("Composed thumbnail system instruction", thumbnail_system)
print_block("Thumbnail user prompt", thumbnail_user)

thumbnail_parsed, thumbnail_raw = gemini.generate_structured_debug(
    thumbnail_system,
    thumbnail_user,
    ThumbnailLLMOutput,
    temperature=0.8,
)

print_llm_exchange(
    stage="thumbnail",
    system_instruction=thumbnail_system,
    user_prompt=thumbnail_user,
    response_schema=ThumbnailLLMOutput,
    parsed=thumbnail_parsed,
    raw_text=thumbnail_raw,
    temperature=0.8,
    model_name=settings.gemini_model,
)

final_thumbnail_prompt = thumbnail_parsed.thumbnail_prompt.strip()
print_block("Final thumbnail prompt", final_thumbnail_prompt)

## Stage 5 — Repository save (`POST /api/v1/repository/save`)

**Dry-run by default** — set `COMMIT_TO_SHEET = True` to append a row.

In [ ]:
# Cell 7 — Repository payload preview / optional save
COMMIT_TO_SHEET = False  # flip to True only when you want to write to Google Sheets

idea_summary = selected_concept.get("video_idea") or " — ".join(
    [
        selected_concept.get("concept_name", ""),
        selected_concept.get("noise_stack", ""),
        f"{selected_concept.get('duration_hours')}h · {selected_concept.get('use_case')}",
    ]
).strip(" —")

repository_payload = {
    "video_idea": idea_summary,
    "title": selected_title,
    "description": final_description,
    "tags": final_tags,
    "thumbnail": final_thumbnail_prompt,
}

print_banner("Stage 5 · Repository payload")
print_block("Row that would be appended", repository_payload)

if COMMIT_TO_SHEET:
    sheets.append_new_video(repository_payload)
    print("\nCommitted row to Google Sheet.")
else:
    print("\nDry run only — no row written. Set COMMIT_TO_SHEET = True to append.")

## Optional — Re-run a single stage with custom inputs

Use this cell to tweak one stage without rerunning the full notebook.

In [ ]:
# Cell 8 — Single-stage sandbox
STAGE = "ideas"  # ideas | titles | metadata | thumbnail
CUSTOM_VIBE = "10h ADHD focus, blue+violet flicker, zero-distraction study field"

stage_map = {
    "ideas": (load_stage_prompt("ideas"), f"Vibe request: {CUSTOM_VIBE}\n\nRepository Video Idea examples (match this prose style; do not repeat these mixes):\n{sheets.video_idea_knowledgebase(history)}", IdeasLLMOutput, 0.9),
    "titles": (load_stage_prompt("titles"), f"Selected video idea:\n{json.dumps(selected_concept, indent=2)}\n\nRepository Title examples:\n{sheets.title_knowledgebase(history)}", TitlesLLMOutput, 0.85),
    "metadata": (load_stage_prompt("metadata"), f"Selected concept:\n{json.dumps(selected_concept, indent=2)}\n\nApproved title: {selected_title}\n\nRepository history descriptions for formatting style transfer:\n{history_context}", MetadataLLMOutput, 0.7),
    "thumbnail": (load_stage_prompt("thumbnail"), json.dumps({"idea": selected_concept, "title": selected_title, "description": final_description, "tags": final_tags}, indent=2), ThumbnailLLMOutput, 0.8),
}

system_prompt, user_prompt, schema, temp = stage_map[STAGE]
parsed, raw = gemini.generate_structured_debug(system_prompt, user_prompt, schema, temperature=temp)
if VERBOSE:
    print_llm_exchange(
        stage=f"sandbox:{STAGE}",
        system_instruction=system_prompt,
        user_prompt=user_prompt,
        response_schema=schema,
        parsed=parsed,
        raw_text=raw,
        temperature=temp,
        model_name=settings.gemini_model,
    )
else:
    print_block("Sandbox parsed output", parsed)